# Anomaly Detection, Semi-Supervised, Self-Supervised, Active, Transfer and Online Learning

Notebook for **Chapter 14** of the *ml* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ml/ch14.html

Tip: change a number, predict the output, then run the cell again.

## 14.1 Anomaly Detection by Statistics: z-score, Robust z, IQR and Control Charts

## 14.2 Anomaly Detection with Machine Learning

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(0)
n = 1000
dist = rng.normal(50, 10, n); mins = 100 + 2.0*(dist-50) + rng.normal(0, 8, n)   # raw units, converted below
Xn = np.c_[mins/2.5, dist/10]                       # columns: minutes, km
Xa = np.vstack([np.c_[rng.normal(300,30,5)/2.5, rng.normal(60,10,5)/10],
                np.c_[rng.normal(150,5,15)/2.5, rng.normal(30,3,15)/10]])
X = np.vstack([Xn, Xa]); y = np.r_[np.zeros(n), np.ones(20)]
Xs = StandardScaler().fit_transform(X)
def rep(name, score):
    top = np.argsort(-score)[:20]
    print(f"{name:18s} AUC={roc_auc_score(y,score):.3f}  precision@20={y[top].mean():.2f}")
iso = IsolationForest(n_estimators=200, random_state=0).fit(Xs); rep("IsolationForest", -iso.score_samples(Xs))
lof = LocalOutlierFactor(n_neighbors=20).fit(Xs);               rep("LOF", -lof.negative_outlier_factor_)
oc  = OneClassSVM(gamma="scale", nu=0.02).fit(Xs);              rep("OneClassSVM", -oc.score_samples(Xs))
ee  = EllipticEnvelope(contamination=0.02, random_state=0).fit(Xs); rep("EllipticEnvelope", -ee.score_samples(Xs))
rep("max |z| per column", np.abs(Xs).max(1))

## 14.3 Semi-Supervised, Self-Supervised and Weak Supervision

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import LabelPropagation, SelfTrainingClassifier
X, y = make_moons(600, noise=0.1, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, random_state=4)
rng = np.random.default_rng(4)       # keep 5 labelled rows per class; -1 marks "unlabelled"
lab = np.r_[rng.choice(np.where(ytr == 0)[0], 5, replace=False), rng.choice(np.where(ytr == 1)[0], 5, replace=False)]
yp = np.full(len(ytr), -1); yp[lab] = ytr[lab]
sup = LogisticRegression().fit(Xtr[lab], ytr[lab])
lp  = LabelPropagation(kernel="knn", n_neighbors=7).fit(Xtr, yp)
st  = SelfTrainingClassifier(LogisticRegression(), threshold=0.9).fit(Xtr, yp)
print("supervised on 10 labels acc", round(sup.score(Xte, yte), 3))
print("LabelPropagation acc", round(lp.score(Xte, yte), 3), "| transduction acc on unlabelled train", round((lp.transduction_[yp == -1] == ytr[yp == -1]).mean(), 3))
print("SelfTraining(logreg) acc", round(st.score(Xte, yte), 3))

## 14.4 Active Learning

## 14.5 Transfer Learning

## 14.6 Online and Incremental Learning, plus Other Paradigms

## 14.7 Isolation Forest by Hand, and Why LOF and Isolation Forest Disagree

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
x = np.r_[rng.normal(10, 1.0, 255), 18.0]          # 255 ordinary values and one outlier, n = 256

def path_length(v, data, depth=0, limit=8):
    """Depth at which value v is isolated by random cuts (stops when alone or at the depth limit)."""
    if len(data) <= 1 or depth >= limit: return depth + (c(len(data)) if len(data) > 1 else 0)
    lo, hi = data.min(), data.max()
    if lo == hi: return depth
    cut = rng.uniform(lo, hi)
    side = data[data < cut] if v < cut else data[data >= cut]
    return path_length(v, side, depth + 1, limit)

def c(n):                                           # average path length of an unsuccessful BST search
    return 0.0 if n <= 1 else 2 * (np.log(n - 1) + 0.5772156649) - 2 * (n - 1) / n

n = len(x)
print("c(256) = %.3f" % c(n))
for name, v in [("outlier 18.0", 18.0), ("typical 10.0", x[0]), ("typical 10.5", x[1])]:
    E = np.mean([path_length(v, x) for _ in range(300)])
    print("%-13s mean path length %.2f  anomaly score 2^(-E/c) = %.3f" % (name, E, 2 ** (-E / c(n))))

c(256) = 10.245
outlier 18.0  mean path length 1.78  anomaly score 2^(-E/c) = 0.886
typical 10.0  mean path length 12.81  anomaly score 2^(-E/c) = 0.420
typical 10.5  mean path length 13.27  anomaly score 2^(-E/c) = 0.407


In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
rng = np.random.default_rng(0)
C = np.array([[1, 0.9], [0.9, 1]]); Xn = rng.multivariate_normal([0, 0], C, 600)
Xa = np.array([[2.0, -2.0], [-1.5, 1.8], [3.2, 3.2]])        # two contextual, one extreme
iso = IsolationForest(n_estimators=300, random_state=0).fit(Xn)
lof = LocalOutlierFactor(n_neighbors=30, novelty=True).fit(Xn)
si, sl = -iso.score_samples(Xn), -lof.score_samples(Xn)
print("99th percentile of normal scores: IF %.3f  LOF %.2f" % (np.percentile(si, 99), np.percentile(sl, 99)))
for p, a, b in zip(Xa, -iso.score_samples(Xa), -lof.score_samples(Xa)):
    print("point %-12s IF %.3f  LOF %.2f   above the 99th percentile: IF %s, LOF %s" % (p, a, b, a > np.percentile(si, 99), b > np.percentile(sl, 99)))

99th percentile of normal scores: IF 0.682  LOF 2.07
point [ 2. -2.]    IF 0.678  LOF 6.35   above the 99th percentile: IF False, LOF True
point [-1.5  1.8]  IF 0.643  LOF 5.18   above the 99th percentile: IF False, LOF True
point [3.2 3.2]    IF 0.752  LOF 2.37   above the 99th percentile: IF True, LOF True


## 14.8 Anomalies in a Metric Over Time: Seasonality, Spikes and Level Shifts

Ordinary z-scores fail on a metric with a weekly rhythm: the weekend dip is "anomalous" every week and a real spike hides inside weekday variation. The recipe is: (1) remove the expected pattern, (2) standardise the residual with a robust spread, (3) use a point rule for spikes and a cumulative rule (CUSUM) for sustained shifts.

In [ ]:
import numpy as np
rng = np.random.default_rng(3)
days = 140; dow = np.arange(days) % 7
season = np.array([100, 104, 108, 110, 112, 80, 70.0])          # weekly pattern of a daily metric
y = season[dow] + rng.normal(0, 2.0, days)
y[60] += 25                                                      # one-day spike
y[100:] += 9                                                     # level shift from day 100
# 1) remove seasonality with the median of each weekday over the first 56 days (all normal)
profile = np.array([np.median(y[:56][dow[:56] == d]) for d in range(7)])
resid = y - profile[dow]
mad = np.median(np.abs(resid[:56] - np.median(resid[:56])))
sigma = 1.4826 * mad
print("robust sigma of residual: %.2f" % sigma)
z = resid / sigma
flag = np.where(np.abs(z) > 4)[0]
print("days flagged before the shift (|robust z| > 4 on the residual):", flag[flag < 100], "z of day 60: %.1f, z of day 9: %.1f" % (z[60], z[9]))
raw = (y - y.mean()) / y.std()
print("plain z of the raw series on day 60: %.2f, largest plain |z| on any day before 100: %.2f" % (raw[60], np.abs(raw[:100]).max()))
# 2) CUSUM on the residual (clipped at 3 sigma so one spike cannot trigger it): detects the level shift
k, h = 0.5 * sigma, 5 * sigma
r = np.clip(resid, -3 * sigma, 3 * sigma)
s = 0.0; alarm = None
for t in range(56, days):
    s = max(0.0, s + r[t] - k)
    if s > h and alarm is None: alarm = t
print("CUSUM alarm at day", alarm, "(true shift starts at day 100)")

robust sigma of residual: 1.35
days flagged before the shift (|robust z| > 4 on the residual): [ 9 60] z of day 60: 15.2, z of day 9: 4.6
plain z of the raw series on day 60: 2.08, largest plain |z| on any day before 100: 2.16
CUSUM alarm at day 100 (true shift starts at day 100)


## 14.9 Hidden Markov Models: Forward Algorithm and Viterbi on Three States

In [ ]:
import numpy as np
states = ["Good", "Worn", "Failing"]; obs_names = ["Low", "Med", "High"]
pi = np.array([0.6, 0.3, 0.1])
A = np.array([[0.7, 0.2, 0.1], [0.1, 0.7, 0.2], [0.0, 0.2, 0.8]])
B = np.array([[0.7, 0.2, 0.1], [0.2, 0.6, 0.2], [0.1, 0.3, 0.6]])
obs = [0, 1, 2]                                                   # Low, Med, High
# forward algorithm
alpha = pi * B[:, obs[0]]
print("alpha1", np.round(alpha, 4))
for t in obs[1:]:
    alpha = (alpha @ A) * B[:, t]
    print("alpha%d" % (t + 1), np.round(alpha, 4))
print("P(Low, Med, High) = %.5f" % alpha.sum())
# Viterbi
delta = pi * B[:, obs[0]]; back = []
print("delta1", np.round(delta, 4))
for t in obs[1:]:
    cand = delta[:, None] * A                                     # cand[i, j] = delta_i * A_ij
    back.append(cand.argmax(0)); delta = cand.max(0) * B[:, t]
    print("delta%d" % (t + 1), np.round(delta, 4), "best previous", [states[i] for i in back[-1]])
path = [int(delta.argmax())]
for b in reversed(back): path.append(int(b[path[-1]]))
print("best path:", [states[i] for i in reversed(path)], "prob %.5f" % delta.max())

alpha1 [0.42 0.06 0.01]
alpha2 [0.06   0.0768 0.0186]
alpha3 [0.005  0.0139 0.0217]
P(Low, Med, High) = 0.04061
delta1 [0.42 0.06 0.01]
delta2 [0.0588 0.0504 0.0126] best previous ['Good', 'Good', 'Good']
delta3 [0.0041 0.0071 0.006 ] best previous ['Good', 'Worn', 'Worn']
best path: ['Good', 'Worn', 'Worn'] prob 0.00706


## 14.10 The Kalman Filter: A One-Dimensional Worked Update

In [ ]:
import numpy as np
# 1-D Kalman filter for a slowly drifting level. x_t = x_(t-1) + process noise (variance Q); z_t = x_t + noise (variance R)
Q, R = 0.1, 1.0
x, P = 0.0, 1.0
for z in (1.2, 0.8):
    P_pred = P + Q; K = P_pred / (P_pred + R)
    x = x + K * (z - x); P = (1 - K) * P_pred
    print("z=%.1f  predicted var %.4f  gain %.4f  estimate %.4f  new var %.4f" % (z, P_pred, K, x, P))
# simulation: filter versus raw measurements
rng = np.random.default_rng(1)
n = 200; truth = np.cumsum(rng.normal(0, np.sqrt(Q), n)); z = truth + rng.normal(0, np.sqrt(R), n)
x, P, est = 0.0, 1.0, []
for zt in z:
    Pp = P + Q; K = Pp / (Pp + R); x = x + K * (zt - x); P = (1 - K) * Pp; est.append(x)
est = np.array(est)
print("RMSE raw measurement: %.3f   RMSE Kalman: %.3f   steady-state gain: %.3f" % (np.sqrt(np.mean((z - truth) ** 2)), np.sqrt(np.mean((est - truth) ** 2)), K))

z=1.2  predicted var 1.1000  gain 0.5238  estimate 0.6286  new var 0.5238
z=0.8  predicted var 0.6238  gain 0.3842  estimate 0.6944  new var 0.3842
RMSE raw measurement: 0.900   RMSE Kalman: 0.371   steady-state gain: 0.270


## Chapter recap

## Practice set

**1. (easy)** Data: `10, 12, 11, 13, 12, 90`. Find the median and MAD. **Answer:** Sorted: 10, 11, 12, 12, 13, 90; median = (12 + 12) / 2 = 12. Absolute deviations: 2, 0, 1, 1, 1, 78 sorted 0, 1, 1, 1, 2, 78; MAD = (1 + 1) / 2 = 1.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ml/ch14.html